# FarmPulse - CNN baseline

Cette expérience fournit une référence simple avant EfficientNet. Les résultats seront comparés dans MLflow.

In [3]:
from pathlib import Path
import sys, json, torch, mlflow
ROOT = Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from torch import nn
from torch.utils.data import DataLoader
from src.models.baseline import BaselineCNN
from src.models.train import ManifestDataset, evaluate

processed = ROOT / 'data' / 'processed'
classes = json.loads((processed / 'classes.json').read_text())
epochs, batch_size = 5, 32

In [5]:
train_loader = DataLoader(ManifestDataset(processed / 'train.csv', 224, True), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(ManifestDataset(processed / 'val.csv', 224, False), batch_size=batch_size)
model = BaselineCNN(len(classes))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
history = []
mlflow.set_tracking_uri(f"sqlite:///{(ROOT / 'mlflow.db').as_posix()}")
with mlflow.start_run(run_name='notebook-baseline-cnn'):
    mlflow.log_params({'architecture': 'baseline_cnn', 'epochs': epochs, 'batch_size': batch_size})
    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            optimizer.zero_grad(); loss = criterion(model(images), labels); loss.backward(); optimizer.step()
        metrics = evaluate(model, val_loader, torch.device('cpu')); history.append(metrics)
        mlflow.log_metrics(metrics, step=epoch)
        print(epoch + 1, metrics)


2026/11/19 18:44:32 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/19 18:44:32 INFO mlflow.store.db.utils: Updating database tables


1 {'accuracy': 0.5272844688408137, 'f1_weighted': 0.48112424241233864}
2 {'accuracy': 0.634807878592186, 'f1_weighted': 0.6075804816042257}
3 {'accuracy': 0.6880852437843074, 'f1_weighted': 0.6742409618684799}
4 {'accuracy': 0.7181143041653213, 'f1_weighted': 0.709827372916827}
5 {'accuracy': 0.7391023571197933, 'f1_weighted': 0.7360797734051087}


Interprétation: ce score est la baseline. EfficientNet doit améliorer au minimum le F1 weighted et l'accuracy sur le même split.

In [6]:
torch.save({'state_dict': model.state_dict(), 'classes': classes}, ROOT / 'models' / 'baseline.pt')

In [8]:
models_dir = ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "state_dict": model.state_dict(),
        "classes": classes
    },
    models_dir / "baseline.pt"
)

with open(models_dir / "baseline_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("Baseline model saved successfully!")
print("Final metrics:", history[-1])

Baseline model saved successfully!
Final metrics: {'accuracy': 0.7391023571197933, 'f1_weighted': 0.7360797734051087}


In [9]:
print("Checkpoint exists:", (ROOT / "models" / "baseline.pt").exists())
print("History exists:", (ROOT / "models" / "baseline_history.json").exists())

Checkpoint exists: True
History exists: True
